# Bibliotecas

## Sistema (os, makedirs, glob)

In [ ]:
from os import makedirs
import os
import glob
from os import listdir

import dotenv
def load_dotenv():
    """Load the .env file normally and also from the current directory."""
    dotenv.load_dotenv()
    dotenv.load_dotenv(dotenv.find_dotenv(usecwd=True))

load_dotenv()

## Básicos (numpy, math, display, locale, time, random, re)

In [2]:
# !python -m pip install jupyter


# !python -m pip install IPython
from IPython.display import display

import math

# !python -m pip install numpy
import numpy as np

import locale
# locale.setlocale(locale.LC_ALL, "pt_BR.UTF-8")  # Use "" for auto, or force e.g. to "en_US.UTF-8"

import time
from datetime import datetime, timedelta, date

from pandas.tseries.offsets import BDay # para os dias úteis
# today = datetime.datetime.today()
# print(today - BDay(4)) # 4 dias úteis atrás

# # !python -m pip install random
import random
random.seed(42)

# !python -m pip install regex
import re

## Leitura e análise de dados (Excel, Pandas, Spark)

In [3]:
# !python -m pip install findspark

# !python -m pip install openpyxl
# import openpyxl

# !python -m pip install xlsxwriter
# import xlsxwriter

# !python -m pip install xlrd
# import xlrd

# !python -m pip install python-calamine
# import python_calamine


# !python -m pip install pandas
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
# pd.set_option('display.float_format', lambda x: '%.2f' % x)

## Finanças (yfinance, mplfinance)

In [4]:
# https://pypi.org/project/yfinance/
# https://github.com/ranaroussi/yfinance/wiki/Ticker

!python -m pip install yfinance
import yfinance as yf

!python -m pip install mplfinance
import mplfinance as mpf

# # Em R
# # https://cran.r-project.org/web/packages/BatchGetSymbols/index.html


[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Funções

## _criar_regressao_bd_acao_

In [5]:
def criar_regressao_bd_acao(
    bd_acao,
    coluna = "Close",
    print_variaveis = False,
    plot_grafico = False,
    tamanho_figsize = (10,5),
    rotacao = 45,
    titulo = "Pontos no fechamento da ação",
    var_pular_final_de_semana_feriados = False,
    ):

    # !python -m pip install scikit-learn
    from sklearn.linear_model import LinearRegression

    # !python -m pip install matplotlib
    import matplotlib.pyplot as plt
    from matplotlib import ticker
    import matplotlib.dates as mdates


    bd_acao_coluna = bd_acao.reset_index()[[bd_acao.index.name, coluna]]

    X = bd_acao_coluna.reset_index()["index"].array.reshape(-1, 1)
    y = bd_acao_coluna.loc[:, coluna].array.reshape(-1, 1)

    modelo_linear = LinearRegression()
    modelo_linear.fit(X, y)

    bd_acao_coluna.loc[:, "Regressão"] = modelo_linear.predict(X)
    # bd_acao_fim

    desvio_padrao = bd_acao_coluna[coluna].std()
    media = bd_acao_coluna[coluna].mean()
    alfa = modelo_linear.coef_[0][0]
    valor_fechamento = bd_acao_coluna.sort_index(ascending = False).iloc[0][coluna]

    # margem = 0.005
    margem = desvio_padrao/media

    if print_variaveis == True:
        print("Média: " + "{:.4f}".format(media))
        print("Desvio padrão: " + "{:.4f}".format(desvio_padrao))
        print("Desvio padrão (%): " + "{:.4f}".format(margem))
        print("Inclinação da reta (alfa, coeficiente angular): " + "{:.4f}".format(alfa))
        print("Valor de fechamento (R$): " + "{:.2f}".format(valor_fechamento))


    if plot_grafico == True:
        plt.figure(figsize = tamanho_figsize)

        ax = plt.gca()

        if var_pular_final_de_semana_feriados == True:
            ax.xaxis.set_major_locator(ticker.LinearLocator(len(bd_acao_coluna)))
            # ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
            # ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt = "%d/%m/%y"))

            plt.xticks(rotation = rotacao)

            plt.scatter(x = bd_acao_coluna[bd_acao.index.name].dt.strftime("%d/%m/%y").astype(str), y = coluna, data = bd_acao_coluna, edgecolors='black', facecolors='none')

            plt.plot(bd_acao_coluna[bd_acao.index.name].dt.strftime("%d/%m/%y").astype(str), bd_acao_coluna["Regressão"]-desvio_padrao, color = "blue", linestyle='dashed')
            plt.plot(bd_acao_coluna[bd_acao.index.name].dt.strftime("%d/%m/%y").astype(str), bd_acao_coluna["Regressão"], color='green')
            plt.plot(bd_acao_coluna[bd_acao.index.name].dt.strftime("%d/%m/%y").astype(str), bd_acao_coluna["Regressão"]+desvio_padrao, color = "blue", linestyle='dashed')

            # ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt = "%d/%m/%y"))
            # ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))

        else:
            ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))

            plt.xticks(bd_acao_coluna[bd_acao.index.name], rotation = rotacao)

            plt.scatter(x = bd_acao_coluna[bd_acao.index.name], y = coluna, data = bd_acao_coluna, edgecolors='black', facecolors='none')

            plt.plot(bd_acao_coluna[bd_acao.index.name], bd_acao_coluna["Regressão"]-desvio_padrao, color = "blue", linestyle='dashed')
            plt.plot(bd_acao_coluna[bd_acao.index.name], bd_acao_coluna["Regressão"], color='green')
            plt.plot(bd_acao_coluna[bd_acao.index.name], bd_acao_coluna["Regressão"]+desvio_padrao, color = "blue", linestyle='dashed')


        plt.title(titulo)
        plt.show()

    return [bd_acao_coluna, media, desvio_padrao, modelo_linear, valor_fechamento]

## _processar_regressao_e_lista_de_indicadores_

In [6]:
def processar_regressao_e_lista_de_indicadores(
    bd_acao,
    coluna_analise,
    qtd_dias,
):

    [_, _, _, modelo_linear, _] = criar_regressao_bd_acao(
        bd_acao,
        coluna = coluna_analise,
        print_variaveis = False,
        plot_grafico = False,
        # titulo = acao + " (fechamento dos últimos " + str(qtd_dias) + " dias)",
        # tamanho_figsize = (15, 6),
        # rotacao = 60,
    )

    lista_regressao_e_indicadores = []
    lista_regressao_e_indicadores_cabecalho = []
    
    lista_regressao_e_indicadores.append(modelo_linear.coef_[0][0]) # 
    lista_regressao_e_indicadores_cabecalho.append("Alfa HLC; últimos "  + str(qtd_dias) + " dias")
        
    # bd_lista_acoes_analise["Taxa média de Remuneração (R$)"] = bd_lista_acoes_analise["Preço fechamento (R$)"]*bd_lista_acoes_analise["Alfa"]
    # bd_lista_acoes_analise["Taxa média de Remuneração (pp/R$, " + str(qtd_dias) +" dias)"] = \
    # bd_lista_acoes_analise["Alfa (" + str(qtd_dias) +" dias)"]/bd_lista_acoes_analise["Preço fechamento (R$, " + str(qtd_dias) +" dias)"]

    return lista_regressao_e_indicadores_cabecalho, lista_regressao_e_indicadores

## _candle_plot_

In [7]:
def candle_plot( # Usa o navegador para criar o gráfico interativo
    dados,
    volume = True,
    mav = np.nan,
    colors = ["orange", "yellow", "blue"],
    titulo = "",
    ):
  
    # !python -m pip install plotly
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    # # !python -m pip install nbformat
    # import nbformat as mc
    # import importlib

    # importlib.reload(mc)
    # %load_ext autoreload
    # %autoreload 2

    if volume == True:
        fig = make_subplots(
            rows = 2,
            cols = 1,
            shared_xaxes = True,
            vertical_spacing = 0.1,
            subplot_titles = ("Candlesticks", "Volume transacionado"),
            row_width = [0.2, 0.7]
        )
    else:
        fig = make_subplots(
            rows = 1,
            cols = 1,
            shared_xaxes = True,
            vertical_spacing = 0.1,
            subplot_titles = ("Candlesticks"),
            row_width = [0.2, 0.7]
        )

    fig.add_trace(go.Candlestick(x=dados.index,
                        open = dados['Open'],
                        high = dados['High'],
                        low = dados['Low'],
                        close = dados['Close']),
                    row = 1, col = 1)

    if mav is not np.nan:
        for i in range(len(mav)):
        # print(i)
            dados["Close "+ str(mav[i]) +" dias"] = dados["Close"].rolling(window=mav[i]).mean()
            fig.add_trace(go.Scatter(x=dados.index,
                            y = dados["Close "+ str(mav[i]) +" dias"],
                            mode = "lines",
                            name = "Média móvel fechamento " + str(mav[i]) + " dias",
                            marker=dict(color=colors[i])),
                        row = 1, col = 1)

    if volume == True:
        fig.add_trace(go.Bar(x=dados.head(60).index,
                            y = dados['Volume'],
                            name = "Volume"),
                    row = 2, col = 1)


    fig.update_layout(
        yaxis_title = "Preço",
        xaxis_rangeslider_visible=False,
        title=titulo,
        )

    fig.show()

## _plota_candlestick_acha_martelos_

In [8]:
def plota_candlestick_acha_martelos(
    bd_acao,
    # periodo = "21d",
    # intervalo = "1d",
    taxa_máxima_para_ser_martelo = 0.2,
  ):
    
    # bd_acao = yf.Ticker(acao).history(
        # period = periodo,
        # interval = intervalo
    # )
    bd_acao["Amplitude Open-Close"] = abs(bd_acao["Open"] - bd_acao["Close"])
    bd_acao["Amplitude High-Low"] = abs(bd_acao["High"] - bd_acao["Low"])

    # taxa_máxima_para_ser_martelo = 0.2
    bd_acao["Martelo?"] = (bd_acao["Amplitude Open-Close"] < taxa_máxima_para_ser_martelo * bd_acao["Amplitude High-Low"])
    # display(bd_acao)

    # bd_acao.loc[:, "Tipo Martelo"] = ""
    bd_acao.loc[(bd_acao["Martelo?"] == True) * (bd_acao["Open"] > bd_acao["Close"]), "Tipo Martelo"] = "Descida"
    bd_acao.loc[(bd_acao["Martelo?"] == True) * (bd_acao["Open"] <= bd_acao["Close"]), "Tipo Martelo"] = "Subida"
    # display(bd_acao)

    lista_datas_martelo = bd_acao[bd_acao["Martelo?"] == True].sort_index(ascending = False).index.to_list()
    string_datas_martelo = ""

    lista_tipos_martelo = bd_acao[bd_acao["Martelo?"] == True].sort_index(ascending = False)["Tipo Martelo"].to_list()
    string_tipos_martelo = ""

    for data in lista_datas_martelo:
        string_datas_martelo = data.strftime("%Y-%m-%d") + ", " + string_datas_martelo

    for tipo in lista_tipos_martelo:
        string_tipos_martelo = tipo + ", " + string_tipos_martelo


    return [bd_acao, string_datas_martelo, string_tipos_martelo]
    # return [bd_acao, lista_datas_martelo, lista_tipos_martelo]
    # return [bd_acao, string_datas_martelo, lista_tipos_martelo]

## _reforma_formato_original_

In [9]:
def reforma_formato_original(
    bd,
    acao,
):
    # acao = bd.iloc[0].loc["Ticker"]
    # display(acao)

    lista_cols_Alfa = [col for col in bd.columns if col.startswith("Alfa")]
    # display(lista_cols_Alfa)

    lista_cols_Martelos = []
    for contador, col in enumerate(bd.columns.str.contains("Martelos")):
        if col == True:
            lista_cols_Martelos.append(bd.columns[contador])
    # display(lista_cols_Martelos)

    # display(bd.drop(lista_cols_Alfa, axis = 1).drop(lista_cols_Martelos, axis = 1))
    bd_acao_reformada = pd.DataFrame(bd.loc[bd["Ticker"] == acao].iloc[:, 4:].drop(lista_cols_Alfa, axis = 1).drop(lista_cols_Martelos, axis = 1)).T.reset_index()
    # display(bd_acao_reformada)

    bd_acao_reformada["Data"] = bd_acao_reformada["index"].str.split("; ").str[0]
    bd_acao_reformada["Indicador"] = bd_acao_reformada["index"].str.split("; ").str[1]
    bd_acao_reformada = bd_acao_reformada.drop("index", axis = 1)
    # display(bd_acao_reformada)

    bd_acao_reformada.columns = ["Valores", "Data", "Indicador"]
    bd_acao_reformada = pd.pivot(
        index = "Data",
        columns = "Indicador",
        values = "Valores",
        data = bd_acao_reformada
    )
    display(bd_acao_reformada)

    # bd_acao_reformada = pd.DataFrame(
        # index = bd_acao_reformada.index.str.split("; "),
        # data = [bd_acao_reformada.index.str.split("; ")]
    # )

    return bd_acao_reformada#.columns


# reforma_formato_original(
#     bd_novo_historico,
#     bd_novo_historico.iloc[0].loc["Ticker"],
# )

# Ler arquivo

## Lista de Ações Tratada

In [28]:
print(os.environ.get('var_caminho_fonte') )

C:\Users\ricardopeloi\OneDrive - falconi365\Data Science\O_Mais_Novo_Day_Trader_do_Brasil\o_mais_novo_day_trader_do_brasil


In [29]:
#f
var_caminho_pasta = os.environ.get('var_caminho_fonte') + r'\Bases'

var_arquivo_lista_acoes_tratada = r"\Lista de ações Tratada.xlsx"

bd_arquivo_original = pd.read_excel(var_caminho_pasta + var_arquivo_lista_acoes_tratada).set_index("Ticker")

bd_arquivo_original.sort_values([col for col in bd_arquivo_original if col.startswith("Volume no último dia útil")], ascending = False).head(5)

,Nome da Empresa,Volume no último dia útil,Data da leitura (último dia útil),industry,sector,fullTimeEmployees,dividendRate,dividendYield,exDividendDate,payoutRatio,...,ebitdaMargins,operatingMargins,epsTrailingTwelveMonths,epsForward,epsCurrentYear,priceEpsCurrentYear,fiftyDayAverageChange,fiftyDayAverageChangePercent,twoHundredDayAverageChange,twoHundredDayAverageChangePercent
Ticker,,,,,,,,,,,,,,,,,,,,,
AZUL4,Azul,282174900,2025-04-27,Airlines,Industrials,"15,367.00",NaN,NaN,NaN,0.00,...,0.18,0.18,-0.35,-0.96,-1.53,-1.28,-1.52,-0.44,-2.95,-0.60
AZUL4,Azul,282174900,2025-04-28,Airlines,Industrials,"15,367.00",NaN,NaN,NaN,0.00,...,0.18,0.18,-0.35,-0.96,-1.53,-1.11,-1.78,-0.51,-3.21,-0.66
AZUL4,Azul,282174900,2025-04-26,Airlines,Industrials,"15,367.00",NaN,NaN,NaN,0.00,...,0.18,0.18,-0.35,-0.96,-1.53,-1.28,-1.52,-0.44,-2.95,-0.60
AZUL4,Azul,279865900,2025-04-29,Airlines,Industrials,"15,367.00",NaN,NaN,NaN,0.00,...,0.18,0.18,-0.35,-0.96,-1.53,-1.21,-1.59,-0.46,-3.03,-0.62
AZUL4,Azul,267269300,2025-05-01,Airlines,Industrials,"15,367.00",NaN,NaN,NaN,0.00,...,0.18,0.18,-26.32,-0.96,-1.53,-0.96,-1.87,-0.56,-3.33,-0.69


## Lista de ações análise 2025-03-10

In [ ]:
var_arquivo_lista_acoes_analise = r"\Lista de ações Análise 2025-03-10.xlsx"

bd_acoes_analise = pd.read_excel(var_caminho_pasta + var_arquivo_lista_acoes_analise)

bd_acoes_analise.head(5)

,Ticker,Nome da Empresa,Volume no último dia útil (lido em 10/03/2025),Market Cap,Alfa (15 dias),Preço Close há 15 dias,Preço HLC em 10/03/2025,Preço HLC há 15 dias,Alfa (39 dias),Preço Close há 39 dias,Preço HLC há 39 dias,Datas dos martelos
0,HAPV3,Hapvida,97225000,15210568704,-0.04,2.51,2.08,2.45,-0.01,2.45,2.39,"21/01/25, 24/01/25, 10/02/25, 13/02/25, 17/02/..."
1,B3SA3,B3,41638200,54814535680,-0.09,11.59,10.48,11.50,-0.02,11.09,10.94,"13/01/25, 21/01/25, 24/01/25, 28/01/25, 03/02/..."
2,MGLU3,Magazine Luiza,40884800,6073303552,-0.00,7.38,8.13,7.31,0.01,7.27,7.13,"24/01/25, 04/02/25, 11/02/25, 12/02/25, 27/02/..."
3,COGN3,Cogna,36134200,3024730624,-0.01,1.70,1.66,1.67,0.01,1.39,1.38,"14/01/25, 16/01/25, 22/01/25, 29/01/25, 03/02/..."
4,ABEV3,Ambev,30190400,205837058048,0.17,11.13,13.12,11.11,0.08,11.11,11.07,"14/01/25, 20/01/25, 23/01/25, 07/02/25, 10/02/..."


# Análises

## Teste com uma ação em específico (saiu da lista)
- Em determinada época, meados de março de 2025, os dados da Gol (GOLL4) não estavam sendo puxados. Notei que após 16/03, data em que escrevo, os dados se normalizaram. Contudo, isso foi bastante importante para minha melhoria do código no tratamento de erros e desvios do padrão normal esperado
- Ao mesmo tempo, notei que havia algumas ações com Ticker duplicado (IGTI), e um outro que não estava retornando dados: ITSA3

In [72]:
# acao = "GOLL4"
acao = "ITSA3"
qtd_dias_maximo = 10

bd_acao = yf.Ticker(acao + ".SA").history(
    start = datetime.today() - timedelta(days=qtd_dias_maximo),
    end = datetime.today(),
    interval = "1d"
)

bd_acao
# len(bd_acao)

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-03-06 00:00:00-03:00,9.11,9.23,9.10,9.20,113300,0.00,0.00
2025-03-07 00:00:00-03:00,9.25,9.39,9.14,9.39,200000,0.00,0.00
2025-03-10 00:00:00-03:00,9.42,9.42,9.25,9.32,122500,0.00,0.00
2025-03-11 00:00:00-03:00,9.42,9.42,9.18,9.30,98000,0.00,0.00
2025-03-12 00:00:00-03:00,9.30,9.30,9.20,9.25,97600,0.00,0.00
2025-03-13 00:00:00-03:00,9.31,9.34,9.18,9.31,90100,0.00,0.00
2025-03-14 00:00:00-03:00,9.35,9.61,9.33,9.60,137500,0.00,0.00


## Teste com uma ação que funciona

In [73]:
acao = "VIVT3"
qtd_dias_maximo = 10
qtd_dias_minimo_escolhido = 3

bd_acao = yf.Ticker(acao + ".SA").history(
    start = datetime.today() - timedelta(days=qtd_dias_maximo),
    end = datetime.today(),
    interval = "1d"
)

qtd_dias_minimo = max(1, min(len(bd_acao), qtd_dias_maximo, qtd_dias_minimo_escolhido))
display(qtd_dias_minimo)

display(bd_acao)
display(bd_acao.iloc[len(bd_acao) - qtd_dias_minimo])

3

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-03-06 00:00:00-03:00,49.07,49.14,48.09,49.00,3882300,0.00,0.00
2025-03-07 00:00:00-03:00,48.72,49.79,48.07,49.47,3332600,0.00,0.00
2025-03-10 00:00:00-03:00,49.39,49.89,48.90,49.89,1592800,0.00,0.00
2025-03-11 00:00:00-03:00,49.89,49.90,48.64,48.94,1804700,0.00,0.00
2025-03-12 00:00:00-03:00,48.95,49.32,48.66,49.29,1756500,0.00,0.00
2025-03-13 00:00:00-03:00,49.09,50.11,48.25,49.73,1548700,0.00,0.00
2025-03-14 00:00:00-03:00,50.28,51.36,50.03,51.16,2810300,0.00,0.00


Open                  48.95
High                  49.32
Low                   48.66
Close                 49.29
Volume         1,756,500.00
Dividends              0.00
Stock Splits           0.00
Name: 2025-03-12 00:00:00-03:00, dtype: float64

## Adicionando linhas via listas na _bd_original_

In [74]:
display(bd_arquivo_original.iloc[0].to_list())

# display(bd_arquivo_original.set_index("Ticker").head())
acao = "B3SA3"
display([acao] + bd_arquivo_original.loc[acao].to_list())
# display(bd_arquivo_original.columns)

['B3',
 np.int64(126021900),
 'Financial Data & Stock Exchanges',
 'financial-data-stock-exchanges',
 'Financial Data & Stock Exchanges',
 'Financial Services',
 'financial-services',
 'Financial Services',
 np.float64(nan),
 np.float64(0.3),
 np.float64(2.32),
 np.float64(1735776000.0),
 np.float64(0.4376),
 np.float64(0.679),
 np.float64(15.506024),
 np.float64(13.83871),
 np.int64(0),
 np.int64(0),
 np.int64(46142515),
 np.int64(58217175),
 np.int64(58217175),
 np.int64(67275739136),
 np.float64(7.0716314),
 np.float64(10.7574),
 np.float64(10.87575),
 np.float64(0.298),
 np.float64(0.025689656),
 np.float64(0.48106),
 np.float64(0.83),
 np.float64(0.93),
 '3:1',
 np.float64(1621209600.0),
 np.float64(7.182),
 np.float64(10.83),
 np.float64(0.04126215),
 np.float64(0.09506309),
 np.float64(0.064034),
 np.float64(1735776000.0),
 np.float64(1.92857),
 'buy',
 np.float64(14.0),
 np.float64(13298551808.0),
 np.float64(2.526),
 np.float64(6308825088.0),
 np.float64(13846761472.0),
 np.fl

['B3SA3',
 'B3',
 np.int64(126021900),
 'Financial Data & Stock Exchanges',
 'financial-data-stock-exchanges',
 'Financial Data & Stock Exchanges',
 'Financial Services',
 'financial-services',
 'Financial Services',
 np.float64(nan),
 np.float64(0.3),
 np.float64(2.32),
 np.float64(1735776000.0),
 np.float64(0.4376),
 np.float64(0.679),
 np.float64(15.506024),
 np.float64(13.83871),
 np.int64(0),
 np.int64(0),
 np.int64(46142515),
 np.int64(58217175),
 np.int64(58217175),
 np.int64(67275739136),
 np.float64(7.0716314),
 np.float64(10.7574),
 np.float64(10.87575),
 np.float64(0.298),
 np.float64(0.025689656),
 np.float64(0.48106),
 np.float64(0.83),
 np.float64(0.93),
 '3:1',
 np.float64(1621209600.0),
 np.float64(7.182),
 np.float64(10.83),
 np.float64(0.04126215),
 np.float64(0.09506309),
 np.float64(0.064034),
 np.float64(1735776000.0),
 np.float64(1.92857),
 'buy',
 np.float64(14.0),
 np.float64(13298551808.0),
 np.float64(2.526),
 np.float64(6308825088.0),
 np.float64(13846761472.

In [75]:
display(bd_arquivo_original.head())
acao = "B3SA3"

display(bd_arquivo_original.loc[acao])
display(bd_arquivo_original.loc[acao].to_list())

# [acao] + bd_arquivo_original.loc[acao].to_list()
# display([acao] + bd_arquivo_original.loc[acao].to_list())
# display(lista_acao)
# display(lista_regressao_e_indicadores_minimo)
# display(lista_regressao_e_indicadores_maximo)
# display([string_datas_martelo, string_tipos_martelo])

,Nome da Empresa,Volume no último dia útil (lido em 16/03/2025),industry,industryKey,industryDisp,sector,sectorKey,sectorDisp,fullTimeEmployees,dividendRate,...,ebitdaMargins,operatingMargins,epsTrailingTwelveMonths,epsForward,epsCurrentYear,priceEpsCurrentYear,fiftyDayAverageChange,fiftyDayAverageChangePercent,twoHundredDayAverageChange,twoHundredDayAverageChangePercent
Ticker,,,,,,,,,,,,,,,,,,,,,
B3SA3,B3,126021900,Financial Data & Stock Exchanges,financial-data-stock-exchanges,Financial Data & Stock Exchanges,Financial Services,financial-services,Financial Services,NaN,0.30,...,0.66,0.63,0.83,0.93,0.89,14.49,2.11,0.20,1.99,0.18
HAPV3,Hapvida,111755000,Insurance - Life,insurance-life,Insurance - Life,Financial Services,financial-services,Financial Services,NaN,NaN,...,0.11,0.44,-0.07,0.29,0.18,12.34,-0.10,-0.04,-1.15,-0.34
NTCO3,Natura,105384100,Household & Personal Products,household-personal-products,Household & Personal Products,Consumer Defensive,consumer-defensive,Consumer Defensive,NaN,0.74,...,0.10,0.04,-2.31,0.94,1.06,8.96,-3.36,-0.26,-4.52,-0.32
COGN3,Cogna,102051600,Education & Training Services,education-training-services,Education & Training Services,Consumer Defensive,consumer-defensive,Consumer Defensive,"24,187.00",NaN,...,0.33,0.27,-0.21,0.18,0.23,7.53,0.28,0.19,0.31,0.21
MGLU3,Magazine Luiza,65687600,Specialty Retail,specialty-retail,Specialty Retail,Consumer Cyclical,consumer-cyclical,Consumer Cyclical,NaN,NaN,...,0.05,0.04,0.61,0.61,0.56,17.14,2.53,0.36,-0.09,-0.01


Nome da Empresa                                                                 B3
Volume no último dia útil (lido em 16/03/2025)                           126021900
industry                                          Financial Data & Stock Exchanges
industryKey                                         financial-data-stock-exchanges
industryDisp                                      Financial Data & Stock Exchanges
                                                                ...               
priceEpsCurrentYear                                                          14.49
fiftyDayAverageChange                                                         2.11
fiftyDayAverageChangePercent                                                  0.20
twoHundredDayAverageChange                                                    1.99
twoHundredDayAverageChangePercent                                             0.18
Name: B3SA3, Length: 68, dtype: object

['B3',
 np.int64(126021900),
 'Financial Data & Stock Exchanges',
 'financial-data-stock-exchanges',
 'Financial Data & Stock Exchanges',
 'Financial Services',
 'financial-services',
 'Financial Services',
 np.float64(nan),
 np.float64(0.3),
 np.float64(2.32),
 np.float64(1735776000.0),
 np.float64(0.4376),
 np.float64(0.679),
 np.float64(15.506024),
 np.float64(13.83871),
 np.int64(0),
 np.int64(0),
 np.int64(46142515),
 np.int64(58217175),
 np.int64(58217175),
 np.int64(67275739136),
 np.float64(7.0716314),
 np.float64(10.7574),
 np.float64(10.87575),
 np.float64(0.298),
 np.float64(0.025689656),
 np.float64(0.48106),
 np.float64(0.83),
 np.float64(0.93),
 '3:1',
 np.float64(1621209600.0),
 np.float64(7.182),
 np.float64(10.83),
 np.float64(0.04126215),
 np.float64(0.09506309),
 np.float64(0.064034),
 np.float64(1735776000.0),
 np.float64(1.92857),
 'buy',
 np.float64(14.0),
 np.float64(13298551808.0),
 np.float64(2.526),
 np.float64(6308825088.0),
 np.float64(13846761472.0),
 np.fl

In [76]:
def detectar_martelos_todos_os_tickers(qtd_dias_maximo = 55, qtd_dias_minimo = 13):
    # qtd_dias_maximo = int(np.round(55/7*5))
    # qtd_dias_minimo = int(np.round(21/7*5))
    
    import pandas as pd
    from datetime import datetime, timedelta

    # acao = "VIVT3"
    # acoes = ["VIVT3"]
    acoes = ["ABEV3"]
    # acoes = ["VIVT3", "CLSA3", "ITSA3", "IGTI3"]
    qtd_dias_maximo = 10
    qtd_dias_minimo = 3
    coluna_analise = "HLC"

    print("=== LENDO DADOS DE HISTÓRICO ===")
    for contador_acoes, acao in enumerate(acoes):
        print(acao)
        if contador_acoes == 0:
            bd_acoes = pd.DataFrame()

        try:
            bd_acao = yf.Ticker(acao + ".SA").history(
                start = datetime.today() - timedelta(days=qtd_dias_maximo),
                end = datetime.today(),
                interval = "1d"
            )

            if len(bd_acao) != 0:
                bd_acao[coluna_analise] = (bd_acao["High"] + bd_acao["Low"] + bd_acao["Close"])/3
                bd_acao["Ticker"] = acao
                bd_acoes = pd.concat([bd_acoes, bd_acao])
                # bd_acao = bd_acao.drop("Ticker", axis = 1)
                # display(bd_acao)

                if contador_acoes == 0:
                    lista_datas = bd_acao.index.strftime("%Y-%m-%d").to_list()
                    # display(lista_datas)

                    lista_cabecalho = []
                    lista_acoes = []
                
                lista_acao = []

                for counter, data in enumerate(lista_datas):
                    # display(data)
                    
                    if contador_acoes == 0:
                        lista_cabecalho = lista_cabecalho + [data + '; ' + s for s in bd_acao.columns]
                    
                    lista_acao = lista_acao + bd_acao.iloc[counter].to_list()
                # display(lista_acao)
                # display(len(lista_acao))

                lista_regressao_e_indicadores_cabecalho_minimo, lista_regressao_e_indicadores_minimo = processar_regressao_e_lista_de_indicadores(
                    bd_acao.iloc[-(qtd_dias_minimo):].copy(),
                    coluna_analise,
                    qtd_dias_minimo
                )
                # display(lista_regressao_e_indicadores_minimo)

                lista_regressao_e_indicadores_cabecalho_maximo, lista_regressao_e_indicadores_maximo = processar_regressao_e_lista_de_indicadores(
                    bd_acao.copy(),
                    coluna_analise,
                    qtd_dias_maximo
                )
                # display(lista_regressao_e_indicadores_maximo)

                [_, string_datas_martelo, string_tipos_martelo] = plota_candlestick_acha_martelos(
                    bd_acao,
                    # periodo = str(qtd_dias_maximo)+"d",
                    # intervalo = "1d",
                    taxa_máxima_para_ser_martelo = 0.2,
                    # display_candlestick = False,
                )
                # display(string_datas_martelo)


                if contador_acoes == 0:
                    lista_cabecalho = bd_arquivo_original.reset_index().columns.to_list() \
                        + lista_cabecalho \
                        + lista_regressao_e_indicadores_cabecalho_minimo \
                        + lista_regressao_e_indicadores_cabecalho_maximo \
                        + ["Martelos", "Tipos de Martelos"]
                    # display(len(lista_cabecalho))
                    # display(lista_cabecalho)

                lista_acao = [acao] + bd_arquivo_original.loc[acao].to_list() \
                        + lista_acao \
                        + lista_regressao_e_indicadores_minimo \
                        + lista_regressao_e_indicadores_maximo \
                        + [string_datas_martelo, string_tipos_martelo]


                lista_acoes.append(lista_acao)
                # display(lista_acoes)

        except:
            # contador_acoes = contador_acoes - 1
            display("erro")

    print("=== FIM DA LEITURA DOS DADOS DE HISTÓRICO ===")

    bd_acao_historico = pd.DataFrame(columns = lista_cabecalho, data = lista_acoes)
    # display(bd_acao_historico)


    bd_acoes = bd_acoes.reset_index()
    bd_acoes["Date"] = bd_acoes["Date"].dt.tz_localize(None)
    bd_acoes = bd_acoes.set_index("Date")
    # bd_acao_historico.to_excel('Bases/Lista de ações Análise ' + datetime.today().strftime("%Y-%m-%d") + '.xlsx')
    # bd_acoes.to_excel('Bases/Base de Dados Histórico.xlsx')


    return bd_acao_historico, bd_acoes


bd_acao_historico, bd_acoes = detectar_martelos_todos_os_tickers()
# bd_acao_historico.to_excel('Temp.xlsx', index = False)
# pd.read_excel("Temp.xlsx")

display(bd_acao_historico)
display(bd_acoes)
# type(bd_acao_historico.loc[0, "Tipos de Martelos"])

=== LENDO DADOS DE HISTÓRICO ===
ABEV3
=== FIM DA LEITURA DOS DADOS DE HISTÓRICO ===


,Ticker,Nome da Empresa,Volume no último dia útil (lido em 16/03/2025),industry,industryKey,industryDisp,sector,sectorKey,sectorDisp,fullTimeEmployees,...,2025-03-14; Close,2025-03-14; Volume,2025-03-14; Dividends,2025-03-14; Stock Splits,2025-03-14; HLC,2025-03-14; Ticker,Alfa HLC; últimos 3 dias,Alfa HLC; últimos 10 dias,Martelos,Tipos de Martelos
0,ABEV3,Ambev,41765100,Beverages - Brewers,beverages-brewers,Beverages - Brewers,Consumer Defensive,consumer-defensive,Consumer Defensive,"43,000.00",...,13.64,41772200,0.00,0.00,13.55,ABEV3,0.28,0.09,,


,Open,High,Low,Close,Volume,Dividends,Stock Splits,HLC,Ticker
Date,,,,,,,,,
2025-03-06,12.77,13.04,12.70,12.85,40524500,0.00,0.00,12.86,ABEV3
2025-03-07,12.78,13.12,12.72,13.10,30190400,0.00,0.00,12.98,ABEV3
2025-03-10,13.00,13.26,12.98,13.12,27718600,0.00,0.00,13.12,ABEV3
2025-03-11,13.08,13.16,12.74,12.86,32659900,0.00,0.00,12.92,ABEV3
2025-03-12,12.90,13.11,12.87,13.03,38219900,0.00,0.00,13.00,ABEV3
2025-03-13,13.08,13.33,12.94,13.33,35316500,0.00,0.00,13.20,ABEV3
2025-03-14,13.38,13.64,13.38,13.64,41772200,0.00,0.00,13.55,ABEV3


### Exportar arquivo de histórico das ações

In [ ]:
bd_acoes = bd_acoes.reset_index()
bd_acoes["Date"] = bd_acoes["Date"].dt.tz_localize(None)
bd_acoes = bd_acoes.set_index("Date")

display(bd_acoes)

#f
bd_acoes.to_excel(os.environ.get('var_caminho_fonte') + r'\Bases\Base de Dados Histórico.xlsx')

,Open,High,Low,Close,Volume,Dividends,Stock Splits,HLC,Ticker
Date,,,,,,,,,
2025-03-06,12.77,13.04,12.70,12.85,40524500,0.00,0.00,12.86,ABEV3
2025-03-07,12.78,13.12,12.72,13.10,30190400,0.00,0.00,12.98,ABEV3
2025-03-10,13.00,13.26,12.98,13.12,27718600,0.00,0.00,13.12,ABEV3
2025-03-11,13.08,13.16,12.74,12.86,32659900,0.00,0.00,12.92,ABEV3
2025-03-12,12.90,13.11,12.87,13.03,38219900,0.00,0.00,13.00,ABEV3
2025-03-13,13.08,13.33,12.94,13.33,35316500,0.00,0.00,13.20,ABEV3
2025-03-14,13.38,13.64,13.38,13.64,41772200,0.00,0.00,13.55,ABEV3


## Teste Candle

In [78]:
primeiro_martelo = bd_acao_historico.loc[0, "Martelos"].split(", ")[0]
display(primeiro_martelo)

if primeiro_martelo != '':
    # display(bd_acao_historico)
    display(bd_acao_historico[bd_acao_historico["Ticker"] == "VIVT3"][[column for column in bd_acao_historico.columns if column.startswith(primeiro_martelo)]])

''

In [79]:
bd_acao = yf.Ticker("VIVT3" + ".SA").history(
        start = datetime.today() - timedelta(days=qtd_dias_maximo),
        end = datetime.today(),
        interval = "1d"
    )

candle_plot( # Usa o navegador para criar o gráfico interativo
    bd_acao,
    # volume = True,
    # mav = np.nan,
    # colors = ["orange", "yellow", "blue"],
    # titulo = "",
)

## Utilizar base gerada com o novo método em 15/03/2025

In [80]:
var_caminho_novo_historico = r"\Lista de ações Análise 2025-03-15.xlsx"

bd_novo_historico = pd.read_excel(var_caminho_pasta + var_caminho_novo_historico)

# display(bd_novo_historico.head(3))
display(len(bd_novo_historico.columns))

var_colunas = [col for col in bd_novo_historico.columns if col.startswith("Alfa")]

lista_qtd_dias = []

for coluna in var_colunas:
    lista_qtd_dias.append(int(coluna.split("últimos")[1].split(" dias")[0]))

bd_novo_historico = bd_novo_historico.sort_values("Alfa HLC; últimos " + str(max(lista_qtd_dias)) + " dias", ascending = False)
# max(lista_qtd_dias)

display(bd_novo_historico.head())

312

,Ticker,Nome da Empresa,Volume no último dia útil (lido em 15/03/2025),Market Cap,2025-01-20; Open,2025-01-20; High,2025-01-20; Low,2025-01-20; Close,2025-01-20; Volume,2025-01-20; Dividends,...,2025-03-14; Low,2025-03-14; Close,2025-03-14; Volume,2025-03-14; Dividends,2025-03-14; Stock Splits,2025-03-14; HLC,Alfa HLC; últimos 13 dias,Alfa HLC; últimos 55 dias,Martelos,Tipos de Martelos
67,EMBR3,Embraer,5653000,55009316864,60.00,60.75,59.55,60.72,1920600,0,...,73.69,74.88,5653000,0.00,0,74.90,1.26,0.38,"2025-01-23, 2025-02-12, 2025-02-17, 2025-02-18...","Descida, Descida, Subida, Descida, Subida, Des..."
213,AMBP3,Ambipar,158100,21726478336,125.20,126.28,120.90,120.90,30700,0,...,128.33,130.44,158200,0.00,0,130.42,1.11,0.19,"2025-01-22, 2025-01-27, 2025-01-28, 2025-02-05...","Descida, Subida, Subida, Subida, Subida, Subid..."
209,TGMA3,Tegma,175800,2274194944,28.49,28.85,28.33,28.60,280900,0,...,33.84,34.49,175800,0.00,0,34.39,0.14,0.15,"2025-01-28, 2025-01-29, 2025-02-07, 2025-02-12...","Descida, Descida, Subida, Subida, Subida, Desc..."
189,FRAS3,Fras-le,333300,7043881984,21.14,21.14,20.75,21.05,113800,0,...,26.00,26.38,333300,0.00,0,26.27,0.25,0.14,"2025-01-22, 2025-01-24, 2025-01-31, 2025-02-07...","Subida, Descida, Subida, Descida, Subida, Desc..."
56,TOTS3,Totvs,6714600,19610107904,28.46,28.70,28.13,28.45,1450300,0,...,33.05,33.47,6791700,0.00,0,33.43,-0.20,0.13,"2025-01-20, 2025-01-22, 2025-01-28, 2025-01-29...","Descida, Descida, Subida, Subida, Subida, Subi..."


### Gráfico do melhor papel

In [81]:
var_caminho_historico_completo = r"\Base de Dados Histórico.xlsx"

bd_historico_completo = pd.read_excel(var_caminho_pasta + var_caminho_historico_completo)
bd_historico_completo

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,HLC,Ticker
0,2025-01-20,10.20,10.46,10.20,10.32,11418100,0.00,0,10.33,B3SA3
1,2025-01-21,10.33,10.48,10.20,10.37,25403800,0.00,0,10.35,B3SA3
2,2025-01-22,10.48,10.50,10.33,10.43,47319900,0.00,0,10.42,B3SA3
3,2025-01-23,10.43,10.50,10.29,10.35,36718200,0.00,0,10.38,B3SA3
4,2025-01-24,10.35,10.40,10.26,10.37,45091300,0.00,0,10.34,B3SA3
...,...,...,...,...,...,...,...,...,...,...
375,2025-03-10,8.90,8.96,8.85,8.96,32292700,0.00,0,8.92,ITSA4
376,2025-03-11,8.98,8.98,8.78,8.87,30812600,0.00,0,8.88,ITSA4
377,2025-03-12,8.90,8.91,8.78,8.84,29533700,0.00,0,8.84,ITSA4
378,2025-03-13,8.85,9.01,8.83,8.98,30594000,0.00,0,8.94,ITSA4


In [82]:
display(bd_novo_historico.iloc[0].loc["Ticker"])

display(bd_novo_historico.iloc[0].loc["Martelos"])
display(bd_novo_historico.iloc[0].loc["Tipos de Martelos"])

# display(bd_acao_historico.iloc[0].loc["Martelos"])
# display(bd_acao_historico.iloc[0].loc["Tipos de Martelos"])

candle_plot( # Usa o navegador para criar o gráfico interativo
    # reforma_formato_original(
    #     bd_novo_historico,
    #     bd_novo_historico.iloc[0].loc["Ticker"],
    # ),
    # bd_acoes[bd_acoes["Ticker"] == bd_acao_historico.iloc[0].loc["Ticker"]]
    bd_historico_completo[bd_historico_completo["Ticker"] == bd_acao_historico.iloc[0].loc["Ticker"]]
    # volume = True,
    # mav = np.nan,
    # colors = ["orange", "yellow", "blue"],
    # titulo = "",
)

'EMBR3'

'2025-01-23, 2025-02-12, 2025-02-17, 2025-02-18, 2025-02-20, 2025-02-25, 2025-02-26, 2025-03-12, 2025-03-14, '

'Descida, Descida, Subida, Descida, Subida, Descida, Subida, Descida, Subida, '

## Verificar mais dados de _info_ da API

In [83]:
# acoes = ["ASAI3", "VAMO3", "BIOM3", "ABEV3"]
acoes = ["ABEV3"]
# acao = acoes[0]

lista_campos_consulta_api = \
    ['industry', 'industryKey', 'industryDisp', 'sector', 'sectorKey', 'sectorDisp', 'fullTimeEmployees', 
     'dividendRate', 'dividendYield', 'exDividendDate', 'payoutRatio', 'beta', 'trailingPE', 'forwardPE', 
     'volume', 'regularMarketVolume', 'averageVolume', 'averageVolume10days', 'averageDailyVolume10Day', 
     'marketCap',
     'priceToSalesTrailing12Months', 'fiftyDayAverage', 'twoHundredDayAverage', 'trailingAnnualDividendRate', 
     'trailingAnnualDividendYield', 'profitMargins', 'trailingEps', 'forwardEps', 'lastSplitFactor', 
     'lastSplitDate', 'enterpriseToRevenue', 'enterpriseToEbitda', '52WeekChange', 'SandP52WeekChange', 
     'lastDividendValue', 'lastDividendDate', 'recommendationMean', 'recommendationKey', 'numberOfAnalystOpinions', 
     'totalCash', 'totalCashPerShare', 'ebitda', 'totalDebt', 'quickRatio', 'currentRatio', 'totalRevenue', 
     'debtToEquity', 'revenuePerShare', 'returnOnAssets', 'returnOnEquity', 'grossProfits', 'freeCashflow', 
     'operatingCashflow', 'earningsGrowth', 'revenueGrowth', 'grossMargins', 'ebitdaMargins', 'operatingMargins', 
     'epsTrailingTwelveMonths', 'epsForward', 'epsCurrentYear', 'priceEpsCurrentYear', 'fiftyDayAverageChange', 
     'fiftyDayAverageChangePercent', 'twoHundredDayAverageChange', 'twoHundredDayAverageChangePercent',]
# print(len(lista_campos_consulta_api))

listas_infos_selecionadas = []
lista_infos_selecionadas = []

for acao in acoes:
    acao_dados = yf.Ticker(acao + ".SA")
    for campo in lista_campos_consulta_api:
        try:
            lista_infos_selecionadas.append(acao_dados.info[campo])
        except:
            lista_infos_selecionadas.append(None)
    
    listas_infos_selecionadas.append(lista_infos_selecionadas)

# display(lista_infos_selecionadas)
display(acao_dados.info["marketCap"])
display(pd.DataFrame(columns = lista_campos_consulta_api, data = listas_infos_selecionadas)["marketCap"])
# type(lista_infos_selecionadas)

213556019200

0    213556019200
Name: marketCap, dtype: int64

In [85]:
var_info_completas = r"\Lista de ações Tratada.xlsx"

bd_info_completas = pd.read_excel(var_caminho_pasta + var_info_completas).set_index("Ticker")
bd_info_completas

,Nome da Empresa,Volume no último dia útil (lido em 16/03/2025),industry,industryKey,industryDisp,sector,sectorKey,sectorDisp,fullTimeEmployees,dividendRate,...,ebitdaMargins,operatingMargins,epsTrailingTwelveMonths,epsForward,epsCurrentYear,priceEpsCurrentYear,fiftyDayAverageChange,fiftyDayAverageChangePercent,twoHundredDayAverageChange,twoHundredDayAverageChangePercent
Ticker,,,,,,,,,,,,,,,,,,,,,
B3SA3,B3,126021900,Financial Data & Stock Exchanges,financial-data-stock-exchanges,Financial Data & Stock Exchanges,Financial Services,financial-services,Financial Services,NaN,0.30,...,0.66,0.63,0.83,0.93,0.89,14.49,2.11,0.20,1.99,0.18
HAPV3,Hapvida,111755000,Insurance - Life,insurance-life,Insurance - Life,Financial Services,financial-services,Financial Services,NaN,NaN,...,0.11,0.44,-0.07,0.29,0.18,12.34,-0.10,-0.04,-1.15,-0.34
NTCO3,Natura,105384100,Household & Personal Products,household-personal-products,Household & Personal Products,Consumer Defensive,consumer-defensive,Consumer Defensive,NaN,0.74,...,0.10,0.04,-2.31,0.94,1.06,8.96,-3.36,-0.26,-4.52,-0.32
COGN3,Cogna,102051600,Education & Training Services,education-training-services,Education & Training Services,Consumer Defensive,consumer-defensive,Consumer Defensive,"24,187.00",NaN,...,0.33,0.27,-0.21,0.18,0.23,7.53,0.28,0.19,0.31,0.21
MGLU3,Magazine Luiza,65687600,Specialty Retail,specialty-retail,Specialty Retail,Consumer Cyclical,consumer-cyclical,Consumer Cyclical,NaN,NaN,...,0.05,0.04,0.61,0.61,0.56,17.14,2.53,0.36,-0.09,-0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ALPA3,Alpargatas,25500,Footwear & Accessories,footwear-accessories,Footwear & Accessories,Consumer Cyclical,consumer-cyclical,Consumer Cyclical,NaN,0.10,...,0.04,-0.25,0.15,NaN,NaN,NaN,0.78,0.12,-0.12,-0.02
INEP3,Inepar,25400,Specialty Industrial Machinery,specialty-industrial-machinery,Specialty Industrial Machinery,Industrials,industrials,Industrials,NaN,NaN,...,0.00,-33.80,11.23,NaN,NaN,NaN,0.05,0.03,-0.18,-0.11
ENGI4,Energisa,24400,Utilities - Regulated Electric,utilities-regulated-electric,Utilities - Regulated Electric,Utilities,utilities,Utilities,"17,141.00",0.58,...,0.26,0.18,1.07,NaN,NaN,NaN,0.25,0.04,-0.32,-0.04
